# exp1 R1 1 rpeak timing

**Self-contained Colab notebook.** Run the setup cells once (top to bottom), then the experiment. The four shared modules are written to `/content` automatically, so you do NOT need to upload a `shared/` folder. Edit Drive paths only in the `config_paths.py` cell below.

> Runtime: set **Runtime -> Change runtime type -> GPU** for the inference and training experiments.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import sys
if '/content' not in sys.path:
    sys.path.insert(0, '/content')
print('Drive mounted; /content on path.')

In [ ]:
# scikit-image backs metrics.calc_ssim; guarantee it for every notebook
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'scikit-image'])
print('scikit-image ready')

In [ ]:
%%writefile /content/config_paths.py
"""
Central path + constant configuration for all JBHI revision Colab experiments.

Derived verbatim from the author's existing notebooks:
  - COLAB_IEEE_FIGURES.ipynb      (paper baselines + EPHNOGRAM chunks + split)
  - cross_dataset_validation.ipynb (SensSmart-Tech + Trail-4 generator)

Every experiment script imports from this file so paths live in ONE place.
If your Drive layout differs, edit ONLY this file.
"""

import os
import torch

# ============================================================================
# >>> EDIT THIS BLOCK IF YOUR DRIVE MOVES <<<
# Updated 2026-07-14: the dataset is now a SHARED folder mounted at
# /content/drive/MyDrive/JBHI-Revision/October Thesis Dataset. If it moves again,
# change ONLY DRIVE_ROOT and OCT_DIR. NOTE: DATA is READ from OCT_DIR; everything
# the code WRITES (retrained checkpoints, outputs) goes under your own Drive
# (OUT_ROOT), so a read-only shared dataset still works.
# ============================================================================
DRIVE_ROOT = "/content/drive/MyDrive"
# Shared dataset root: contains "Refine-Chunked", "Best-epoch-trail-4", etc. (read-only OK).
OCT_DIR    = f"{DRIVE_ROOT}/JBHI-Revision/October Thesis Dataset"

# ----------------------------------------------------------------------------
# EPHNOGRAM data (training-domain test set = 453 segments, 3 subjects)
# .npz keys per chunk: 'ecg', 'pcg', 'subject_id', 'scenario'
#   ecg, pcg : float32 arrays, 64000 samples (8 s @ 8 kHz), RAW (not normalized)
#   subject_id : e.g. 'S009', 'S019', 'S023'
#   scenario   : free-text, e.g. 'rest', 'exercise bicycle stress test'
# ----------------------------------------------------------------------------
EPH_CHUNKS_DIR = f"{OCT_DIR}/Refine-Chunked"
# Subject split. keys: train/val/test_subjects. IF THIS FILE IS ABSENT the
# notebooks fall back to an arbitrary split, which gives the WRONG test set
# (the 2026-07-10 run hit this: 22 segments / 4 subjects instead of the paper's
# 453 segments / 3 subjects). Upload the real split JSON, or point this at it.
SPLIT_FILE     = f"{EPH_CHUNKS_DIR}/train_val_test_split.json"

# ----------------------------------------------------------------------------
# Model checkpoints  --  CONFIRM ALL FOUR EXIST IN THE NEW ACCOUNT
#
#   "Proposed" MUST be the Trail-4 generator (polarity fix, BatchNorm output),
#   i.e. trail4_best.pt or trail4_generator.pth. A "trial2_*" / Trail-2 file is
#   the WRONG (Tanh) model: it loads only partially under strict=False and
#   produces a near-flat, invalid reconstruction (see the 2026-07-10 run).
#
#   The three baselines lived under a SEPARATE top-level folder
#   (Paper1_ECG_Reconstruction) in the old account. Confirm they were migrated;
#   if they now sit under OCT_DIR, repoint them below.
# ----------------------------------------------------------------------------
CKPT = {
    # Baselines are RETRAINED by exp7 and SAVED to your own writable Drive
    # (DRIVE_ROOT/JBHI_Revision/baseline_checkpoints == OUT_ROOT/baseline_checkpoints),
    # so a read-only shared dataset is fine. Same paths are used to load them later.
    "CNN":      f"{DRIVE_ROOT}/JBHI_Revision/baseline_checkpoints/CNN_EncoderDecoder/best_model.pt",
    "UNet":     f"{DRIVE_ROOT}/JBHI_Revision/baseline_checkpoints/UNet_Baseline/best_model.pt",
    "WGAN-GP":  f"{DRIVE_ROOT}/JBHI_Revision/baseline_checkpoints/WGAN_GP/best_model.pt",
    "Proposed": f"{OCT_DIR}/Best-epoch-trail-4/trail4_best.pt",   # read from the shared dataset
}
CKPT_PROPOSED_ALT = f"{OCT_DIR}/trail4_generator.pth"   # fallback copy (state_dict)

# ----------------------------------------------------------------------------
# SensSmart-Tech cross-dataset (32 subjects, 338 records @ 1 kHz -> 8 kHz)
#   WFDB pairs: {base}_ecg.hea/.dat and {base}_pcg.hea/.dat ; ECG lead II
# ----------------------------------------------------------------------------
SENSSMART_WFDB  = f"{OCT_DIR}/senssmarttech-database/WFDB"
SENSSMART_PREP  = f"{OCT_DIR}/milestone2_preprocessed"     # cached .npz (keys ecg/pcg)

# ----------------------------------------------------------------------------
# Output root for ALL revision artifacts (created on demand)
#   Each experiment writes to a named sub-folder here.
# ----------------------------------------------------------------------------
OUT_ROOT = f"{DRIVE_ROOT}/JBHI_Revision"

def out_dir(name):
    """Return (and create) an output sub-folder under OUT_ROOT."""
    p = os.path.join(OUT_ROOT, name)
    os.makedirs(p, exist_ok=True)
    return p

# ----------------------------------------------------------------------------
# Signal + inference constants
# ----------------------------------------------------------------------------
FS          = 8000        # sampling rate (Hz)
SEG_LEN     = 64000       # samples per segment (8 s)
NUM_SCENARIOS = 3         # rest / exercise / recovery one-hot
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# R-peak detector settings (identical to metrics.ipynb; keep fixed for reproducibility)
RPEAK_DISTANCE   = 400    # min inter-peak samples = 50 ms @ 8 kHz
RPEAK_PROMINENCE = 0.5

# SensSmart preprocessing bands (from cross_dataset_validation.ipynb)
ECG_BAND = (0.5, 40.0)
PCG_BAND = (25.0, 400.0)
FILTER_ORDER = 4

# Scenario one-hots
SCEN_REST     = [1.0, 0.0, 0.0]
SCEN_EXERCISE = [0.0, 1.0, 0.0]
SCEN_RECOVERY = [0.0, 0.0, 1.0]


def scenario_to_index(scenario_str):
    """Map EPHNOGRAM free-text scenario -> class index {0,1,2}.

    Matches the mapping used in COLAB_IEEE_FIGURES.ipynb.
    """
    s = str(scenario_str).lower()
    if "rest" in s or "laying" in s:
        return 0
    if "exercise" in s or "bicycle" in s or "treadmill" in s or "running" in s:
        return 1
    return 2

In [ ]:
%%writefile /content/models.py
"""
Model architectures for the JBHI revision experiments.

Reproduced VERBATIM from the author's COLAB_IEEE_FIGURES.ipynb (paper Table II
baselines) and models.ipynb (ablation trails), so the checkpoints in
config_paths.CKPT load cleanly.

Which class loads which checkpoint
----------------------------------
    CKPT['CNN']      -> CNNEncoderDecoder()        # no scenario, forward(x)
    CKPT['UNet']     -> UNetGenerator()            # pix2pix, forward(x)
    CKPT['WGAN-GP']  -> UNetGenerator()            # same generator as U-Net
    CKPT['Proposed'] -> ImprovedGeneratorNoTanh()  # scenario, forward(x, s_onehot)

The proposed model's forward takes a scenario one-hot; the three baselines take
only the PCG. Every inference script must branch on this (helper `run_generator`
below does it for you).

For the R1.4 component ablation, use `ConfigurableGenerator`, which exposes the
three grouped factors as switches: use_attention, use_dilated, and
output_activation in {'batchnorm','tanh','linear'}.

NOTE ON PARAM COUNTS: the submitted paper's numbers (CNN 9.8M, U-Net 13.6M,
Proposed 14.0M) do NOT match these classes (which give ~larger counts). Always
report count_params(model_loaded_from_checkpoint) computed here -- see R2.2/R2.5.
"""

import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


# ===========================================================================
# Shared bottleneck modules
# ===========================================================================
class MultiHeadSelfAttention(nn.Module):
    """8-head self-attention over the temporal axis at the bottleneck."""
    def __init__(self, channels, num_heads=8, dropout=0.1):
        super().__init__()
        assert channels % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = channels // num_heads
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Conv1d(channels, channels * 3, kernel_size=1, bias=False)
        self.proj = nn.Conv1d(channels, channels, kernel_size=1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, C, T = x.shape
        qkv = self.qkv(x).reshape(B, 3, self.num_heads, self.head_dim, T)
        qkv = qkv.permute(1, 0, 2, 4, 3)          # (3, B, heads, T, head_dim)
        # .contiguous() is REQUIRED for SDPA to select the flash / memory-efficient
        # kernel (O(T) memory). Without it the permuted views fall back to the MATH
        # kernel, which materializes the T x T score matrix -- that OOM-ed the
        # attention variants even on a 40 GB A100 at batch 48. Mathematically
        # identical to softmax(QK^T/sqrt(d))V, so existing checkpoints are unaffected.
        q, k, v = qkv[0].contiguous(), qkv[1].contiguous(), qkv[2].contiguous()
        p = self.dropout.p if self.training else 0.0
        out = F.scaled_dot_product_attention(q, k, v, dropout_p=p)
        out = out.transpose(2, 3).reshape(B, C, T)
        return self.proj(out) + x


class MultiScaleDilatedBlock(nn.Module):
    """Parallel dilated convs at rates 1, 2, 4 with a 1x1 fusion + residual."""
    def __init__(self, channels):
        super().__init__()
        c1 = channels // 3 + (1 if channels % 3 >= 1 else 0)
        c2 = channels // 3 + (1 if channels % 3 >= 2 else 0)
        c3 = channels // 3
        self.conv_d1 = nn.Conv1d(channels, c1, 3, padding=1, dilation=1)
        self.conv_d2 = nn.Conv1d(channels, c2, 3, padding=2, dilation=2)
        self.conv_d4 = nn.Conv1d(channels, c3, 3, padding=4, dilation=4)
        self.fusion = nn.Sequential(
            nn.Conv1d(channels, channels, 1),
            nn.BatchNorm1d(channels), nn.ReLU(inplace=True))

    def forward(self, x):
        cat = torch.cat([self.conv_d1(x), self.conv_d2(x), self.conv_d4(x)], dim=1)
        return self.fusion(cat) + x


class EnhancedResidualBlock(nn.Module):
    """Residual block with optional MHSA and MSDC (Trails 2-4)."""
    def __init__(self, channels, use_attention=False, use_dilated=True):
        super().__init__()
        self.conv1 = nn.Conv1d(channels, channels, 3, padding=1)
        self.bn1 = nn.BatchNorm1d(channels)
        self.use_dilated = use_dilated
        if use_dilated:
            self.dilated_block = MultiScaleDilatedBlock(channels)
        self.conv2 = nn.Conv1d(channels, channels, 3, padding=1)
        self.bn2 = nn.BatchNorm1d(channels)
        self.use_attention = use_attention
        if use_attention:
            self.attention = MultiHeadSelfAttention(channels, num_heads=8)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        residual = x
        out = self.relu(self.bn1(self.conv1(x)))
        if self.use_dilated:
            out = self.dilated_block(out)
        out = self.bn2(self.conv2(out))
        if self.use_attention:
            out = self.attention(out)
        return self.relu(out + residual)


# ===========================================================================
# Paper Table II baseline: CNN Encoder-Decoder (8-level, no skips, no scenario)
# ===========================================================================
class ConvBlock(nn.Module):
    def __init__(self, i, o, k=4, s=2, p=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv1d(i, o, k, s, p, bias=False),
            nn.BatchNorm1d(o), nn.LeakyReLU(0.2, inplace=True))
    def forward(self, x): return self.block(x)


class DeconvBlock(nn.Module):
    def __init__(self, i, o, k=4, s=2, p=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.ConvTranspose1d(i, o, k, s, p, bias=False),
            nn.BatchNorm1d(o), nn.ReLU(inplace=True))
    def forward(self, x): return self.block(x)


class CNNEncoderDecoder(nn.Module):
    """Paper CNN baseline -> loads CKPT['CNN']. forward(x)."""
    def __init__(self, in_channels=1, out_channels=1, base_filters=64):
        super().__init__()
        bf = base_filters
        self.enc1 = ConvBlock(in_channels, bf)
        self.enc2 = ConvBlock(bf, bf * 2)
        self.enc3 = ConvBlock(bf * 2, bf * 4)
        self.enc4 = ConvBlock(bf * 4, bf * 8)
        self.enc5 = ConvBlock(bf * 8, bf * 8)
        self.enc6 = ConvBlock(bf * 8, bf * 8)
        self.enc7 = ConvBlock(bf * 8, bf * 8)
        self.enc8 = ConvBlock(bf * 8, bf * 8)
        self.dec1 = DeconvBlock(bf * 8, bf * 8)
        self.dec2 = DeconvBlock(bf * 8, bf * 8)
        self.dec3 = DeconvBlock(bf * 8, bf * 8)
        self.dec4 = DeconvBlock(bf * 8, bf * 8)
        self.dec5 = DeconvBlock(bf * 8, bf * 4)
        self.dec6 = DeconvBlock(bf * 4, bf * 2)
        self.dec7 = DeconvBlock(bf * 2, bf)
        self.final = nn.ConvTranspose1d(bf, out_channels, 4, 2, 1)

    def forward(self, x):
        n = x.size(2)
        e = self.enc1(x); e = self.enc2(e); e = self.enc3(e); e = self.enc4(e)
        e = self.enc5(e); e = self.enc6(e); e = self.enc7(e); e = self.enc8(e)
        d = self.dec1(e); d = self.dec2(d); d = self.dec3(d); d = self.dec4(d)
        d = self.dec5(d); d = self.dec6(d); d = self.dec7(d)
        out = self.final(d)
        if out.size(2) != n:
            diff = n - out.size(2)
            out = F.pad(out, (diff // 2, diff - diff // 2))
        return out


# ===========================================================================
# Paper Table II baseline: pix2pix U-Net (shared by U-Net and WGAN-GP)
# ===========================================================================
class UNetBlock(nn.Module):
    def __init__(self, i, o, normalize=True, dropout=0.0):
        super().__init__()
        layers = [nn.Conv1d(i, o, 4, 2, 1, bias=False)]
        if normalize:
            layers.append(nn.BatchNorm1d(o))
        layers.append(nn.LeakyReLU(0.2, inplace=True))
        if dropout:
            layers.append(nn.Dropout(dropout))
        self.block = nn.Sequential(*layers)
    def forward(self, x): return self.block(x)


class UNetUpBlock(nn.Module):
    def __init__(self, i, o, dropout=0.0):
        super().__init__()
        layers = [nn.ConvTranspose1d(i, o, 4, 2, 1, bias=False),
                  nn.BatchNorm1d(o), nn.ReLU(inplace=True)]
        if dropout:
            layers.append(nn.Dropout(dropout))
        self.block = nn.Sequential(*layers)
    def forward(self, x, skip):
        x = self.block(x)
        if x.size(2) != skip.size(2):
            diff = skip.size(2) - x.size(2)
            x = F.pad(x, (diff // 2, diff - diff // 2))
        return torch.cat([x, skip], dim=1)


class UNetGenerator(nn.Module):
    """Paper U-Net / WGAN-GP generator -> loads CKPT['UNet'] or CKPT['WGAN-GP']. forward(x)."""
    def __init__(self, in_channels=1, out_channels=1, features=64):
        super().__init__()
        f = features
        self.e1 = UNetBlock(in_channels, f, normalize=False)
        self.e2 = UNetBlock(f, f * 2); self.e3 = UNetBlock(f * 2, f * 4)
        self.e4 = UNetBlock(f * 4, f * 8); self.e5 = UNetBlock(f * 8, f * 8)
        self.e6 = UNetBlock(f * 8, f * 8); self.e7 = UNetBlock(f * 8, f * 8)
        self.e8 = UNetBlock(f * 8, f * 8, normalize=False)
        self.d1 = UNetUpBlock(f * 8, f * 8, dropout=0.5)
        self.d2 = UNetUpBlock(f * 16, f * 8, dropout=0.5)
        self.d3 = UNetUpBlock(f * 16, f * 8, dropout=0.5)
        self.d4 = UNetUpBlock(f * 16, f * 8)
        self.d5 = UNetUpBlock(f * 16, f * 4); self.d6 = UNetUpBlock(f * 8, f * 2)
        self.d7 = UNetUpBlock(f * 4, f)
        self.final = nn.ConvTranspose1d(f * 2, out_channels, 4, 2, 1)

    def forward(self, x):
        e1 = self.e1(x); e2 = self.e2(e1); e3 = self.e3(e2); e4 = self.e4(e3)
        e5 = self.e5(e4); e6 = self.e6(e5); e7 = self.e7(e6); e8 = self.e8(e7)
        d = self.d1(e8, e7); d = self.d2(d, e6); d = self.d3(d, e5); d = self.d4(d, e4)
        d = self.d5(d, e3); d = self.d6(d, e2); d = self.d7(d, e1)
        out = self.final(d)
        if out.size(2) != x.size(2):
            diff = x.size(2) - out.size(2)
            out = F.pad(out, (diff // 2, diff - diff // 2))
        return out


# ===========================================================================
# Proposed generator (Trail-4): U-Net + MHSA + MSDC + scenario, BatchNorm output
# ===========================================================================
class ImprovedGeneratorNoTanh(nn.Module):
    """Proposed model -> loads CKPT['Proposed']. forward(x, scenario_onehot)."""
    def __init__(self, signal_length=64000, base_channels=64, num_scenarios=3):
        super().__init__()
        bc = base_channels
        self.scenario_embed = nn.Linear(num_scenarios, bc)
        self.enc1 = nn.Sequential(nn.Conv1d(1, bc, 7, 2, 3), nn.BatchNorm1d(bc), nn.ReLU(True))
        self.enc2 = nn.Sequential(nn.Conv1d(bc, bc*2, 5, 2, 2), nn.BatchNorm1d(bc*2), nn.ReLU(True))
        self.enc3 = nn.Sequential(nn.Conv1d(bc*2, bc*4, 5, 2, 2), nn.BatchNorm1d(bc*4), nn.ReLU(True))
        self.enc4 = nn.Sequential(nn.Conv1d(bc*4, bc*8, 5, 2, 2), nn.BatchNorm1d(bc*8), nn.ReLU(True))
        self.enc5 = nn.Sequential(nn.Conv1d(bc*8, bc*8, 5, 2, 2), nn.BatchNorm1d(bc*8), nn.ReLU(True))
        self.bottleneck = nn.Sequential(
            EnhancedResidualBlock(bc*8, use_attention=True, use_dilated=True),
            EnhancedResidualBlock(bc*8, use_attention=True, use_dilated=True),
            EnhancedResidualBlock(bc*8, use_attention=True, use_dilated=True),
            EnhancedResidualBlock(bc*8, use_attention=True, use_dilated=True))
        self.dec5 = nn.Sequential(nn.ConvTranspose1d(bc*8, bc*8, 4, 2, 1), nn.BatchNorm1d(bc*8), nn.ReLU(True))
        self.dec4 = nn.Sequential(nn.Conv1d(bc*16, bc*8, 1), nn.ConvTranspose1d(bc*8, bc*4, 4, 2, 1), nn.BatchNorm1d(bc*4), nn.ReLU(True))
        self.dec3 = nn.Sequential(nn.Conv1d(bc*8, bc*4, 1), nn.ConvTranspose1d(bc*4, bc*2, 4, 2, 1), nn.BatchNorm1d(bc*2), nn.ReLU(True))
        self.dec2 = nn.Sequential(nn.Conv1d(bc*4, bc*2, 1), nn.ConvTranspose1d(bc*2, bc, 4, 2, 1), nn.BatchNorm1d(bc), nn.ReLU(True))
        self.dec1 = nn.Sequential(nn.Conv1d(bc*2, bc, 1), nn.ConvTranspose1d(bc, 1, 4, 2, 1), nn.BatchNorm1d(1))

    def forward(self, x, scenario_onehot):
        s = self.scenario_embed(scenario_onehot).unsqueeze(2)
        e1 = self.enc1(x) + s
        e2 = self.enc2(e1); e3 = self.enc3(e2); e4 = self.enc4(e3); e5 = self.enc5(e4)
        b = self.bottleneck(e5)
        d5 = torch.cat([self.dec5(b), e4], dim=1)
        d4 = torch.cat([self.dec4(d5), e3], dim=1)
        d3 = torch.cat([self.dec3(d4), e2], dim=1)
        d2 = torch.cat([self.dec2(d3), e1], dim=1)
        return self.dec1(d2)


# ===========================================================================
# Configurable generator for the R1.4 component ablation
# ===========================================================================
class ConfigurableGenerator(nn.Module):
    """Ablation generator: toggle each grouped factor independently.

    use_attention : add MHSA in the 4 bottleneck blocks
    use_dilated   : add MSDC in the 4 bottleneck blocks
    output_activation : 'batchnorm' (proposed) | 'tanh' | 'linear'
    use_scenario  : condition on the scenario one-hot (keep True to match proposed)

    Ablation rows the reviewer asked for map to:
        U-Net only          -> use_attention=False, use_dilated=False
        U-Net + MSDC        -> use_attention=False, use_dilated=True
        U-Net + MHSA        -> use_attention=True,  use_dilated=False
        U-Net + both        -> use_attention=True,  use_dilated=True
        Proposed + Tanh     -> both True, output_activation='tanh'
        Proposed + linear   -> both True, output_activation='linear'
        Proposed (BatchNorm)-> both True, output_activation='batchnorm'
    Polarity-on/off is a DATA-loader switch; adversarial-on/off is a TRAINING
    switch -- neither is an architecture change, so control them in the trainer.
    """
    def __init__(self, base_channels=64, num_scenarios=3,
                 use_attention=True, use_dilated=True,
                 output_activation="batchnorm", use_scenario=True):
        super().__init__()
        bc = base_channels
        self.use_scenario = use_scenario
        if use_scenario:
            self.scenario_embed = nn.Linear(num_scenarios, bc)
        self.enc1 = nn.Sequential(nn.Conv1d(1, bc, 7, 2, 3), nn.BatchNorm1d(bc), nn.ReLU(True))
        self.enc2 = nn.Sequential(nn.Conv1d(bc, bc*2, 5, 2, 2), nn.BatchNorm1d(bc*2), nn.ReLU(True))
        self.enc3 = nn.Sequential(nn.Conv1d(bc*2, bc*4, 5, 2, 2), nn.BatchNorm1d(bc*4), nn.ReLU(True))
        self.enc4 = nn.Sequential(nn.Conv1d(bc*4, bc*8, 5, 2, 2), nn.BatchNorm1d(bc*8), nn.ReLU(True))
        self.enc5 = nn.Sequential(nn.Conv1d(bc*8, bc*8, 5, 2, 2), nn.BatchNorm1d(bc*8), nn.ReLU(True))
        self.bottleneck = nn.Sequential(*[
            EnhancedResidualBlock(bc*8, use_attention=use_attention, use_dilated=use_dilated)
            for _ in range(4)])
        self.dec5 = nn.Sequential(nn.ConvTranspose1d(bc*8, bc*8, 4, 2, 1), nn.BatchNorm1d(bc*8), nn.ReLU(True))
        self.dec4 = nn.Sequential(nn.Conv1d(bc*16, bc*8, 1), nn.ConvTranspose1d(bc*8, bc*4, 4, 2, 1), nn.BatchNorm1d(bc*4), nn.ReLU(True))
        self.dec3 = nn.Sequential(nn.Conv1d(bc*8, bc*4, 1), nn.ConvTranspose1d(bc*4, bc*2, 4, 2, 1), nn.BatchNorm1d(bc*2), nn.ReLU(True))
        self.dec2 = nn.Sequential(nn.Conv1d(bc*4, bc*2, 1), nn.ConvTranspose1d(bc*2, bc, 4, 2, 1), nn.BatchNorm1d(bc), nn.ReLU(True))
        head = [nn.Conv1d(bc*2, bc, 1), nn.ConvTranspose1d(bc, 1, 4, 2, 1)]
        if output_activation == "batchnorm":
            head.append(nn.BatchNorm1d(1))
        elif output_activation == "tanh":
            head.append(nn.Tanh())
        elif output_activation == "linear":
            pass
        else:
            raise ValueError(f"unknown output_activation {output_activation}")
        self.dec1 = nn.Sequential(*head)

    def forward(self, x, scenario_onehot=None):
        e1 = self.enc1(x)
        if self.use_scenario and scenario_onehot is not None:
            e1 = e1 + self.scenario_embed(scenario_onehot).unsqueeze(2)
        e2 = self.enc2(e1); e3 = self.enc3(e2); e4 = self.enc4(e3); e5 = self.enc5(e4)
        b = self.bottleneck(e5)
        d5 = torch.cat([self.dec5(b), e4], dim=1)
        d4 = torch.cat([self.dec4(d5), e3], dim=1)
        d3 = torch.cat([self.dec3(d4), e2], dim=1)
        d2 = torch.cat([self.dec2(d3), e1], dim=1)
        return self.dec1(d2)


# ===========================================================================
# Discriminator (for R1.4 adversarial retraining)
# ===========================================================================
class SpectralNormDiscriminator(nn.Module):
    """Spectrally-normalized PatchGAN critic with scenario conditioning."""
    def __init__(self, base_channels=64, num_scenarios=3):
        super().__init__()
        bc = base_channels
        self.scenario_embed = nn.Linear(num_scenarios, bc)
        sn = nn.utils.spectral_norm
        self.conv1 = sn(nn.Conv1d(2, bc, 7, 2, 3))
        self.conv2 = sn(nn.Conv1d(bc, bc*2, 5, 2, 2)); self.bn2 = nn.BatchNorm1d(bc*2)
        self.conv3 = sn(nn.Conv1d(bc*2, bc*4, 5, 2, 2)); self.bn3 = nn.BatchNorm1d(bc*4)
        self.conv4 = sn(nn.Conv1d(bc*4, bc*8, 5, 2, 2)); self.bn4 = nn.BatchNorm1d(bc*8)
        self.conv5 = sn(nn.Conv1d(bc*8, bc*8, 5, 2, 2)); self.bn5 = nn.BatchNorm1d(bc*8)
        self.pool = nn.AdaptiveAvgPool1d(1); self.fc = nn.Linear(bc*8, 1)
        self.lrelu = nn.LeakyReLU(0.2, True)

    def forward(self, pcg, ecg, scenario_onehot):
        x = torch.cat([pcg, ecg], dim=1)
        s = self.scenario_embed(scenario_onehot).unsqueeze(2)
        x = self.lrelu(self.conv1(x)) + s
        x = self.lrelu(self.bn2(self.conv2(x)))
        x = self.lrelu(self.bn3(self.conv3(x)))
        x = self.lrelu(self.bn4(self.conv4(x)))
        x = self.lrelu(self.bn5(self.conv5(x)))
        return torch.sigmoid(self.fc(self.pool(x).squeeze(2)))


# ===========================================================================
# WGAN-GP critic (for retraining the WGAN-GP baseline; extracted verbatim from
# the original 05_GAN_WGAN_GP notebook). No BatchNorm, no sigmoid -> a valid
# WGAN critic. The generator is the plain UNetGenerator above.
# ===========================================================================
class WGANCritic(nn.Module):
    def __init__(self, in_channels=2, features=64):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv1d(in_channels, features, 4, 2, 1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv1d(features, features * 2, 4, 2, 1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv1d(features * 2, features * 4, 4, 2, 1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv1d(features * 4, features * 8, 4, 1, 1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv1d(features * 8, 1, 4, 1, 1),
        )

    def forward(self, x, y):
        return self.model(torch.cat([x, y], dim=1))


def compute_gradient_penalty(critic, pcg, real_ecg, fake_ecg, device):
    """WGAN-GP gradient penalty. Interpolates the ECG only; PCG conditions."""
    bs = real_ecg.size(0)
    alpha = torch.rand(bs, 1, 1, device=device)
    interp = (alpha * real_ecg + (1 - alpha) * fake_ecg).requires_grad_(True)
    c_interp = critic(pcg, interp)
    grads = torch.autograd.grad(
        outputs=c_interp, inputs=interp,
        grad_outputs=torch.ones_like(c_interp),
        create_graph=True, retain_graph=True)[0]
    grads = grads.view(bs, -1)
    return ((grads.norm(2, dim=1) - 1) ** 2).mean()


# ===========================================================================
# Loading + utility helpers
# ===========================================================================
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def load_model(model, checkpoint_path, device="cpu", strict=False):
    """Load a checkpoint into `model`, handling common save formats."""
    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
    if isinstance(ckpt, dict):
        for key in ("generator_state_dict", "model_state_dict", "state_dict"):
            if key in ckpt:
                ckpt = ckpt[key]; break
    state = {k.replace("module.", ""): v for k, v in ckpt.items()} \
            if isinstance(ckpt, dict) else ckpt
    result = model.load_state_dict(state, strict=strict)
    # Loud guard: a large key mismatch means the checkpoint does not fit this
    # architecture (e.g. a Trail-2/trial2 file loaded into the Trail-4 model).
    n_model = max(len(model.state_dict()), 1)
    n_missing, n_unexpected = len(result.missing_keys), len(result.unexpected_keys)
    tag = os.path.basename(checkpoint_path)
    if n_missing or n_unexpected:
        flag = "  <-- LIKELY WRONG CHECKPOINT FOR THIS MODEL" if n_missing > 0.10 * n_model else ""
        print(f"[load_model] {tag}: {n_missing}/{n_model} params missing, "
              f"{n_unexpected} unexpected{flag}")
    else:
        print(f"[load_model] {tag}: exact match ({n_model} tensors).")
    model.to(device).eval()
    return model, result


def build_paper_models(ckpt_dict, device="cpu"):
    """Load all four paper models. Returns {name: model}."""
    models = {}
    models["CNN"], _     = load_model(CNNEncoderDecoder(), ckpt_dict["CNN"], device)
    models["UNet"], _    = load_model(UNetGenerator(), ckpt_dict["UNet"], device)
    models["WGAN-GP"], _ = load_model(UNetGenerator(), ckpt_dict["WGAN-GP"], device)
    models["Proposed"], _= load_model(ImprovedGeneratorNoTanh(), ckpt_dict["Proposed"], device)
    return models


def run_generator(model, pcg_tensor, scenario_tensor, is_proposed):
    """Uniform inference call. `is_proposed` selects the (x, s) signature."""
    with torch.no_grad():
        if is_proposed:
            out = model(pcg_tensor, scenario_tensor)
        else:
            out = model(pcg_tensor)
    return out.squeeze().cpu().numpy()

In [ ]:
%%writefile /content/metrics.py
"""
Metrics for the JBHI revision.

Two families:
  (A) Signal-fidelity metrics  -- identical to the submitted paper's metrics.ipynb
                                  (PSNR, SNR, SSIM, Pearson r, MAE, RMSE,
                                   R-peak AMPLITUDE error, QRS-width error).
  (B) Temporal R-peak DETECTION metrics  -- NEW, added to answer Reviewer 1 #1.
       These detect R-peaks INDEPENDENTLY in the reconstructed ECG and match
       them to ground-truth peaks within a tolerance window, then report
       sensitivity, PPV, F1, FP, FN, timing-error stats, RR-interval error,
       and heart-rate error. This is the metric the paper was missing.

Keep the detector settings fixed (config_paths.RPEAK_*) so every experiment and
the paper agree.
"""

import numpy as np
import scipy.signal as signal
from scipy.stats import pearsonr


# ---------------------------------------------------------------------------
# R-peak detection (single, shared definition)
# ---------------------------------------------------------------------------
def detect_r_peaks(ecg, fs=8000, distance=400, prominence=0.5):
    """scipy find_peaks on an ECG trace. Returns integer sample indices."""
    peaks, _ = signal.find_peaks(np.asarray(ecg).ravel(),
                                 distance=distance, prominence=prominence)
    return peaks


# ---------------------------------------------------------------------------
# (A) Signal-fidelity metrics
# ---------------------------------------------------------------------------
def calc_psnr(real, fake):
    real, fake = np.asarray(real).ravel(), np.asarray(fake).ravel()
    mse = np.mean((real - fake) ** 2)
    if mse < 1e-10:
        return 100.0
    peak = np.max(np.abs(real))
    return float(20 * np.log10(peak / np.sqrt(mse)))


def calc_snr(real, fake):
    real, fake = np.asarray(real).ravel(), np.asarray(fake).ravel()
    noise = real - fake
    return float(10 * np.log10(np.sum(real ** 2) / (np.sum(noise ** 2) + 1e-12)))


def calc_correlation(real, fake):
    real, fake = np.asarray(real).ravel(), np.asarray(fake).ravel()
    try:
        c, _ = pearsonr(real, fake)
        return float(c)
    except Exception:
        return 0.0


def calc_ssim(real, fake):
    from skimage.metrics import structural_similarity as compare_ssim  # lazy: Colab-only dep
    real, fake = np.asarray(real).ravel(), np.asarray(fake).ravel()
    r = (real - real.min()) / (real.max() - real.min() + 1e-8)
    f = (fake - fake.min()) / (fake.max() - fake.min() + 1e-8)
    return float(compare_ssim(r, f, data_range=1.0))


def calc_mae(real, fake):
    return float(np.mean(np.abs(np.asarray(real).ravel() - np.asarray(fake).ravel())))


def calc_rmse(real, fake):
    d = np.asarray(real).ravel() - np.asarray(fake).ravel()
    return float(np.sqrt(np.mean(d ** 2)))


def calc_r_peak_amplitude_error(real, fake, fs=8000, normalize01=True, **kw):
    """Paper's 'R-peak error' -> renamed 'R-peak AMPLITUDE error'.

    Detects R-peaks on the GROUND TRUTH only, then averages the amplitude error at
    those locations. Kept ONLY as a fidelity measure; NOT a timing metric (R1 #1).

    normalize01=True (DEFAULT) min-max normalizes both signals to [0,1] before the
    error, which REPRODUCES the paper's ~0.126 scale (confirmed: the paper's metric
    is [0,1]-normalized). normalize01=False gives the raw z-scored form, which is
    ~8x larger (z-scored ECG spans ~6-8 units vs 1 for [0,1]) -- that is what the
    first exp4/6/7 runs reported (~1.0). Use the default for consistency with the
    manuscript. Returns np.nan when the ground truth has no detectable R-peaks.
    """
    real, fake = np.asarray(real).ravel(), np.asarray(fake).ravel()
    peaks = detect_r_peaks(real, fs=fs, **kw)
    if len(peaks) == 0:
        return np.nan
    if normalize01:
        real = (real - real.min()) / (real.max() - real.min() + 1e-8)
        fake = (fake - fake.min()) / (fake.max() - fake.min() + 1e-8)
    return float(np.mean(np.abs(real[peaks] - fake[peaks])))


def _qrs_width_at_peak(ecg, peak_idx, fs=8000, threshold=0.5, search_ms=25):
    amp = abs(ecg[peak_idx]); thr = threshold * amp
    win = int(search_ms / 1000 * fs)
    q = peak_idx
    for j in range(peak_idx - 1, max(peak_idx - win, 0), -1):
        if abs(ecg[j]) < thr:
            q = j; break
    s = peak_idx
    for j in range(peak_idx + 1, min(peak_idx + win, len(ecg))):
        if abs(ecg[j]) < thr:
            s = j; break
    return (s - q) / fs * 1000.0


def calc_qrs_width_error(real, fake, fs=8000, **kw):
    real, fake = np.asarray(real).ravel(), np.asarray(fake).ravel()
    peaks = detect_r_peaks(real, fs=fs, **kw)
    if len(peaks) == 0:
        return np.nan
    errs = [abs(_qrs_width_at_peak(real, p, fs) - _qrs_width_at_peak(fake, p, fs))
            for p in peaks]
    return float(np.mean(errs))


# ---------------------------------------------------------------------------
# (B) Temporal R-peak DETECTION metrics  (NEW -- Reviewer 1 #1)
# ---------------------------------------------------------------------------
def match_peaks(true_peaks, pred_peaks, tol_samples):
    """Greedy nearest-neighbour matching within a tolerance window.

    Each true peak claims the closest unused predicted peak within tol_samples.
    Returns (matches, TP, FP, FN, signed_timing_errors_samples).
    signed error = pred_index - true_index (positive = reconstructed peak late).
    """
    true_peaks = list(np.asarray(true_peaks).astype(int))
    pred_peaks = list(np.asarray(pred_peaks).astype(int))
    used, matches, terr = set(), [], []
    for tp in true_peaks:
        cands = [(abs(pp - tp), pp) for pp in pred_peaks
                 if pp not in used and abs(pp - tp) <= tol_samples]
        if cands:
            _, pp = min(cands)
            used.add(pp); matches.append((tp, pp)); terr.append(pp - tp)
    TP = len(matches)
    FN = len(true_peaks) - TP
    FP = len(pred_peaks) - TP
    return matches, TP, FP, FN, np.asarray(terr, dtype=float)


def temporal_rpeak_metrics(true_ecg, pred_ecg, fs=8000, tol_ms=50,
                           distance=400, prominence=0.5):
    """Full temporal detection report for ONE segment.

    Returns a dict with detection (Se/PPV/F1/FP/FN), timing-error (ms),
    RR-interval error (ms), and heart-rate error (bpm). Segments with no
    ground-truth peaks return n_true=0 and NaN rates (exclude from means, but
    COUNT them -- a reconstruction that yields peaks where truth has none is a
    real failure; see caller handling).
    """
    tol = int(round(tol_ms / 1000 * fs))
    t_pk = detect_r_peaks(true_ecg, fs, distance, prominence)
    p_pk = detect_r_peaks(pred_ecg, fs, distance, prominence)

    matches, TP, FP, FN, terr = match_peaks(t_pk, p_pk, tol)
    sens = TP / (TP + FN) if (TP + FN) else np.nan
    ppv  = TP / (TP + FP) if (TP + FP) else np.nan
    f1   = (2 * sens * ppv / (sens + ppv)
            if (sens and ppv and not np.isnan(sens) and not np.isnan(ppv)
                and (sens + ppv) > 0) else np.nan)

    terr_ms = np.abs(terr) / fs * 1000.0
    mae_t   = float(np.mean(terr_ms)) if terr_ms.size else np.nan
    med_t   = float(np.median(terr_ms)) if terr_ms.size else np.nan
    iqr_t   = float(np.percentile(terr_ms, 75) - np.percentile(terr_ms, 25)) \
              if terr_ms.size else np.nan

    # RR-interval error on consecutive MATCHED pairs only (avoids missed-beat bias)
    rr_err = np.nan
    if len(matches) >= 2:
        tp_sorted = sorted(matches)
        rr_true, rr_pred = [], []
        for (t0, p0), (t1, p1) in zip(tp_sorted[:-1], tp_sorted[1:]):
            rr_true.append((t1 - t0) / fs * 1000.0)
            rr_pred.append((p1 - p0) / fs * 1000.0)
        rr_err = float(np.mean(np.abs(np.array(rr_true) - np.array(rr_pred))))

    # Heart-rate error (bpm), estimator = 60000 / median-RR
    def _hr(pk):
        if len(pk) < 2:
            return np.nan
        return 60000.0 / np.median(np.diff(pk) / fs * 1000.0)
    hr_true, hr_pred = _hr(t_pk), _hr(p_pk)
    hr_err = float(abs(hr_true - hr_pred)) if not (np.isnan(hr_true) or np.isnan(hr_pred)) else np.nan

    return {
        "n_true": int(len(t_pk)), "n_pred": int(len(p_pk)),
        "TP": TP, "FP": FP, "FN": FN,
        "sensitivity": sens, "ppv": ppv, "f1": f1,
        "mae_timing_ms": mae_t, "median_timing_ms": med_t, "iqr_timing_ms": iqr_t,
        "rr_error_ms": rr_err,
        "hr_true_bpm": hr_true, "hr_pred_bpm": hr_pred, "hr_error_bpm": hr_err,
    }


def aggregate_temporal(rows):
    """Aggregate a list of per-segment temporal_rpeak_metrics dicts.

    Detection totals (TP/FP/FN) are POOLED then Se/PPV/F1 recomputed (the
    correct way to aggregate detection over segments). Timing/RR/HR errors are
    averaged over segments where they are defined; the valid-N is reported so
    the denominator is transparent.
    """
    rows = [r for r in rows if r is not None]
    TP = sum(r["TP"] for r in rows)
    FP = sum(r["FP"] for r in rows)
    FN = sum(r["FN"] for r in rows)
    sens = TP / (TP + FN) if (TP + FN) else np.nan
    ppv  = TP / (TP + FP) if (TP + FP) else np.nan
    f1   = 2 * sens * ppv / (sens + ppv) if (sens + ppv) else np.nan

    def _mean(key):
        v = [r[key] for r in rows if r[key] is not None and not np.isnan(r[key])]
        return (float(np.mean(v)), float(np.std(v)), len(v)) if v else (np.nan, np.nan, 0)

    out = {"n_segments": len(rows), "TP": TP, "FP": FP, "FN": FN,
           "sensitivity": sens, "ppv": ppv, "f1": f1}
    for k in ["mae_timing_ms", "median_timing_ms", "iqr_timing_ms",
              "rr_error_ms", "hr_error_bpm"]:
        m, s, n = _mean(k)
        out[k + "_mean"], out[k + "_std"], out[k + "_validN"] = m, s, n
    return out


# ---------------------------------------------------------------------------
# Preprocessing helpers (shared with the paper pipeline)
# ---------------------------------------------------------------------------
def ensure_positive_polarity(ecg):
    """Flip so R-peaks point up. Applied to the ECG TARGET only (never PCG)."""
    ecg = np.asarray(ecg)
    return -ecg if np.abs(ecg.min()) > np.abs(ecg.max()) else ecg


def normalize_zscore(sig):
    sig = np.asarray(sig, dtype=np.float32)
    return ((sig - sig.mean()) / (sig.std() + 1e-8)).astype(np.float32)

In [ ]:
%%writefile /content/ieee_style.py
"""
IEEE double-column figure style for the JBHI revision.

Design goals (per IEEE Transactions / J-BHI figure rules):
  * Vector output (PDF) + high-DPI PNG (600) for every figure.
  * Serif face (Times) to match IEEE body text; DejaVu Serif fallback on Colab.
  * BOLD axis labels that always carry explicit units.
  * Thick spines and ticks so lines stay legible at column scale.
  * constrained_layout everywhere -> no text/label/legend overlaps.
  * Column widths fixed to IEEE spec: single = 3.5 in, double = 7.16 in.

Usage
-----
    from ieee_style import set_ieee_style, new_fig, finalize_axes, save_fig
    set_ieee_style()
    fig, ax = new_fig("single", height=2.2)
    ax.plot(t, y)
    finalize_axes(ax, xlabel="Time (s)", ylabel="Amplitude (a.u.)")
    save_fig(fig, out_dir, "fig_name")   # writes fig_name.pdf AND fig_name.png
"""

import matplotlib
matplotlib.use("Agg")            # headless-safe; remove if you want inline display
import matplotlib.pyplot as plt

# IEEE column widths (inches)
COL_SINGLE = 3.5
COL_DOUBLE = 7.16

# Colour-blind-safe palette (Okabe-Ito), stable role -> colour mapping
COLORS = {
    "truth":    "#009E73",   # bluish green  -> ground truth
    "proposed": "#0072B2",   # blue          -> proposed model
    "cnn":      "#D55E00",   # vermillion    -> CNN
    "unet":     "#CC79A7",   # reddish purple-> U-Net
    "wgan":     "#E69F00",   # orange        -> WGAN-GP
    "pcg":      "#666666",   # grey          -> PCG input
    "accent":   "#0072B2",
    "muted":    "#999999",
}


def set_ieee_style():
    """Apply the global rcParams. Call once at the top of every script."""
    plt.rcParams.update({
        "figure.dpi":        150,
        "savefig.dpi":       600,
        "savefig.bbox":      "tight",
        "savefig.pad_inches": 0.02,
        "figure.constrained_layout.use": True,
        "font.size":         8,
        "font.family":       "serif",
        "font.serif":        ["Times New Roman", "Times", "DejaVu Serif"],
        "mathtext.fontset":  "dejavuserif",
        "text.usetex":       False,
        "axes.labelsize":    9,
        "axes.labelweight":  "bold",
        "axes.titlesize":    9,
        "axes.titleweight":  "bold",
        "axes.linewidth":    1.1,
        "axes.grid":         True,
        "grid.alpha":        0.25,
        "grid.linewidth":    0.4,
        "xtick.labelsize":   8,
        "ytick.labelsize":   8,
        "xtick.major.width": 1.0,
        "ytick.major.width": 1.0,
        "xtick.major.size":  3.5,
        "ytick.major.size":  3.5,
        "xtick.direction":   "out",
        "ytick.direction":   "out",
        "legend.fontsize":   7.5,
        "legend.framealpha": 0.92,
        "legend.edgecolor":  "0.7",
        "legend.handlelength": 1.6,
        "lines.linewidth":   1.0,
        "lines.solid_capstyle": "round",
    })


def new_fig(width="single", height=2.2, nrows=1, ncols=1, **kw):
    """Create a figure sized to an IEEE column.

    width : 'single' (3.5 in), 'double' (7.16 in), or a float in inches.
    height: figure height in inches.
    Returns (fig, ax) or (fig, axes) exactly like plt.subplots.
    """
    w = {"single": COL_SINGLE, "double": COL_DOUBLE}.get(width, width)
    fig, ax = plt.subplots(nrows, ncols, figsize=(w, height), **kw)
    return fig, ax


def finalize_axes(ax, xlabel=None, ylabel=None, title=None,
                  legend=False, legend_loc="best", legend_ncol=1):
    """Apply bold labels with units, thicken spines, tidy ticks.

    Always pass xlabel/ylabel WITH units, e.g. 'Time (ms)', 'PSNR (dB)'.
    """
    if xlabel:
        ax.set_xlabel(xlabel, fontweight="bold")
    if ylabel:
        ax.set_ylabel(ylabel, fontweight="bold")
    if title:
        ax.set_title(title, fontweight="bold")
    for spine in ax.spines.values():
        spine.set_linewidth(1.1)
    ax.tick_params(axis="both", which="major", width=1.0, labelsize=8)
    if legend:
        leg = ax.legend(loc=legend_loc, ncol=legend_ncol, frameon=True)
        leg.get_frame().set_linewidth(0.6)
    return ax


def save_fig(fig, out_directory, name):
    """Save both a vector PDF (for LaTeX) and a 600-dpi PNG (for preview)."""
    import os
    pdf = os.path.join(out_directory, f"{name}.pdf")
    png = os.path.join(out_directory, f"{name}.png")
    fig.savefig(pdf)
    fig.savefig(png)
    plt.close(fig)
    print(f"  saved  {pdf}")
    print(f"  saved  {png}")
    return pdf, png

In [ ]:
import importlib
import config_paths, models, metrics, ieee_style
for m in (config_paths, models, metrics, ieee_style):
    importlib.reload(m)
ieee_style.set_ieee_style()
print('shared modules ready | device =', config_paths.DEVICE)

## Experiment

# exp1 -- Temporal R-peak detection metrics (Reviewer 1, comment 1)

The submitted paper reports only an **amplitude** R-peak error
(`calc_r_peak_error`, renamed here `calc_r_peak_amplitude_error`). Reviewer 1
asks whether the reconstructed ECG recovers R-peaks at the **right times**.
This script supplies that missing timing metric.

Pipeline for every one of the 453 EPHNOGRAM test segments:
  1. Load the raw `.npz` chunk (keys `ecg`, `pcg`, `subject_id`, `scenario`).
  2. Polarity-fix the ground-truth ECG, then z-score the ECG target and the
     PCG input (exactly the paper pipeline).
  3. Run the Proposed generator, `model(pcg, scenario_onehot)`.
  4. z-score the reconstruction and SAVE it to `out_dir('exp1_reconstructions')`
     as `.npy` (exp3 reuses these waveforms).
  5. Detect R-peaks independently in the ground truth and the reconstruction,
     match them within +/-50 ms and +/-150 ms, and score
     sensitivity, PPV, F1, FP, FN, timing error, RR error, and HR error.
  6. Score a **PCG-only** S1 detector (Shannon-energy envelope of the
     band-passed PCG) against the same ground-truth R-peaks, so the table
     answers "does the reconstruction beat PCG alone?".

Outputs (all under `config_paths.OUT_ROOT`):
  * `exp1_reconstructions/<stem>.npy`         -- z-scored reconstructions
  * `exp1_rpeak_timing/results.json`          -- every number cited by the LaTeX
  * `exp1_rpeak_timing/exp1_temporal_table.tex`     -- main comparison table
  * `exp1_rpeak_timing/exp1_persubject_table.tex`   -- per-subject / per-scenario
  * `exp1_rpeak_timing/fig_exp1_overlay_hist.{pdf,png}`  -- double-column figure
  * `exp1_rpeak_timing/fig_exp1_persubject_f1.{pdf,png}` -- single-column figure

The script is idempotent and resumable: cached reconstructions are reused, and
any missing checkpoint or data path prints a clear note instead of crashing.

In [ ]:
# ---------------------------------------------------------------------------
# Cell 1 -- environment, Drive mount, shared imports
# ---------------------------------------------------------------------------
# NOTE: copy the LOCAL `shared/` folder (config_paths.py, models.py, metrics.py,
# ieee_style.py) to `MyDrive/JBHI_Revision/shared` BEFORE running, or paste the
# four modules into this runtime. The sys.path line below points at that folder.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    print("[note] not in Colab; assuming the shared modules are importable locally.")

import sys
sys.path.append('/content/drive/MyDrive/JBHI_Revision/shared')  # copy shared/ here first

import os
import glob
import json
import numpy as np
import torch
import scipy.signal as sp_signal

import config_paths
import models
import metrics
import ieee_style

try:
    from tqdm.auto import tqdm
except Exception:  # tqdm is optional
    def tqdm(x, **kw):
        return x

ieee_style.set_ieee_style()
import matplotlib.pyplot as plt  # after set_ieee_style so rcParams apply

FS       = config_paths.FS
DIST     = config_paths.RPEAK_DISTANCE      # 400 samples = 50 ms @ 8 kHz
PROM     = config_paths.RPEAK_PROMINENCE    # 0.5
DEVICE   = config_paths.DEVICE
TOLS_MS  = (50, 150)                         # the two tolerance windows scored

RECON_DIR = config_paths.out_dir('exp1_reconstructions')  # feeds exp3
OUT_DIR   = config_paths.out_dir('exp1_rpeak_timing')

print(f"[cfg] device           : {DEVICE}")
print(f"[cfg] reconstructions  : {RECON_DIR}")
print(f"[cfg] results / figures: {OUT_DIR}")

## Cell 2 -- helpers

`temporal_metrics_from_peaks` mirrors `metrics.temporal_rpeak_metrics` but
takes peak-index arrays directly, so the PCG-only S1 detector is scored with
the exact same matching, timing, RR, and HR logic as the reconstruction.
`detect_s1_peaks` is the PCG-only baseline: band-pass, normalized Shannon
energy, smoothing, `find_peaks`, then a systole/diastole heuristic that labels
S1 without ever looking at the ground-truth ECG.

In [ ]:
def temporal_metrics_from_peaks(t_pk, p_pk, fs, tol_ms):
    """Score predicted peaks against true peaks; keys match temporal_rpeak_metrics."""
    tol = int(round(tol_ms / 1000 * fs))
    matches, TP, FP, FN, terr = metrics.match_peaks(t_pk, p_pk, tol)
    sens = TP / (TP + FN) if (TP + FN) else np.nan
    ppv  = TP / (TP + FP) if (TP + FP) else np.nan
    f1   = (2 * sens * ppv / (sens + ppv)
            if (sens and ppv and not np.isnan(sens) and not np.isnan(ppv)
                and (sens + ppv) > 0) else np.nan)
    terr_ms = np.abs(terr) / fs * 1000.0
    mae_t = float(np.mean(terr_ms)) if terr_ms.size else np.nan
    med_t = float(np.median(terr_ms)) if terr_ms.size else np.nan
    iqr_t = (float(np.percentile(terr_ms, 75) - np.percentile(terr_ms, 25))
             if terr_ms.size else np.nan)
    rr_err = np.nan
    if len(matches) >= 2:
        ms = sorted(matches)
        rr_true, rr_pred = [], []
        for (t0, p0), (t1, p1) in zip(ms[:-1], ms[1:]):
            rr_true.append((t1 - t0) / fs * 1000.0)
            rr_pred.append((p1 - p0) / fs * 1000.0)
        rr_err = float(np.mean(np.abs(np.array(rr_true) - np.array(rr_pred))))

    def _hr(pk):
        pk = np.asarray(pk)
        if len(pk) < 2:
            return np.nan
        return 60000.0 / np.median(np.diff(pk) / fs * 1000.0)

    hr_t, hr_p = _hr(t_pk), _hr(p_pk)
    hr_err = float(abs(hr_t - hr_p)) if not (np.isnan(hr_t) or np.isnan(hr_p)) else np.nan
    return {
        "n_true": int(len(t_pk)), "n_pred": int(len(p_pk)),
        "TP": TP, "FP": FP, "FN": FN,
        "sensitivity": sens, "ppv": ppv, "f1": f1,
        "mae_timing_ms": mae_t, "median_timing_ms": med_t, "iqr_timing_ms": iqr_t,
        "rr_error_ms": rr_err,
        "hr_true_bpm": hr_t, "hr_pred_bpm": hr_p, "hr_error_bpm": hr_err,
    }


def _bandpass(x, lo, hi, fs, order):
    ny = 0.5 * fs
    b, a = sp_signal.butter(order, [lo / ny, hi / ny], btype='band')
    return sp_signal.filtfilt(b, a, np.asarray(x, dtype=np.float64))


def detect_s1_peaks(pcg, fs):
    """PCG-only S1 baseline via a normalized Shannon-energy envelope.

    Returns (s1_peaks, envelope). No ground-truth ECG is used at any step.
    """
    lo, hi = config_paths.PCG_BAND
    x = _bandpass(pcg, lo, hi, fs, config_paths.FILTER_ORDER)
    x = x / (np.max(np.abs(x)) + 1e-8)
    se = -(x ** 2) * np.log(x ** 2 + 1e-12)          # normalized Shannon energy
    win = max(1, int(0.04 * fs))                     # 40 ms smoothing window
    env = np.convolve(se, np.ones(win) / win, mode='same')
    env = env / (np.max(np.abs(env)) + 1e-8)
    peaks, _ = sp_signal.find_peaks(env, distance=int(0.12 * fs), prominence=0.10)
    if len(peaks) < 3:
        return peaks, env
    # Systole (S1->S2) is shorter than diastole (S2->S1). Keep the peak set whose
    # leading intervals are the shorter ones; that set is S1.
    d = np.diff(peaks)
    mean_even = float(np.mean(d[0::2])) if len(d[0::2]) else np.inf
    mean_odd  = float(np.mean(d[1::2])) if len(d[1::2]) else np.inf
    s1 = peaks[0::2] if mean_even <= mean_odd else peaks[1::2]
    return s1, env


def _sanitize(obj):
    """Recursively convert numpy scalars and NaN to JSON-native values."""
    if isinstance(obj, dict):
        return {k: _sanitize(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_sanitize(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating, float)):
        return None if np.isnan(obj) else float(obj)
    if isinstance(obj, (np.ndarray,)):
        return [_sanitize(v) for v in obj.tolist()]
    return obj

## Cell 3 -- load the test split and the Proposed generator

The test set is every `.npz` chunk whose `subject_id` is in
`split['test_subjects']`. The Proposed generator loads from
`config_paths.CKPT['Proposed']`; if that file is missing the script still runs
on any reconstructions already cached on disk.

In [ ]:
split_path = config_paths.SPLIT_FILE
chunk_dir  = config_paths.EPH_CHUNKS_DIR

if not os.path.exists(split_path):
    raise SystemExit(f"[abort] split file not found: {split_path}\n"
                     f"        Fix config_paths.SPLIT_FILE to your Drive layout.")

with open(split_path, 'r') as f:
    split = json.load(f)
test_subjects = set(split['test_subjects'])
print(f"[data] test subjects: {sorted(test_subjects)}")

all_npz = sorted(glob.glob(os.path.join(chunk_dir, '*.npz')))
if not all_npz:
    raise SystemExit(f"[abort] no .npz chunks under {chunk_dir}")

test_chunks = []
for path in all_npz:
    try:
        d = np.load(path, allow_pickle=True)
        if str(d['subject_id']) in test_subjects:
            test_chunks.append(path)
    except Exception as e:
        print(f"[warn] could not read {os.path.basename(path)}: {e}")
print(f"[data] test segments: {len(test_chunks)}")

# Load the Proposed generator (guarded: cached reconstructions can stand in).
model = None
ckpt_path = config_paths.CKPT['Proposed']
if os.path.exists(ckpt_path):
    try:
        model, missing = models.load_model(models.ImprovedGeneratorNoTanh(),
                                            ckpt_path, device=DEVICE)
        print(f"[model] loaded Proposed generator "
              f"({models.count_params(model):,} params) from {ckpt_path}")
    except Exception as e:
        print(f"[warn] failed to load checkpoint ({e}); will use cached .npy only.")
elif os.path.exists(config_paths.CKPT_PROPOSED_ALT):
    try:
        model, missing = models.load_model(models.ImprovedGeneratorNoTanh(),
                                            config_paths.CKPT_PROPOSED_ALT, device=DEVICE)
        print(f"[model] loaded Proposed generator from fallback "
              f"{config_paths.CKPT_PROPOSED_ALT}")
    except Exception as e:
        print(f"[warn] fallback load failed ({e}); will use cached .npy only.")
else:
    print(f"[warn] no checkpoint at {ckpt_path}; will use cached .npy only.")

## Cell 4 -- inference + per-segment scoring

For each segment: z-score the ECG target and PCG input, run (or reload) the
reconstruction, cache it, then score both the reconstruction and the PCG-only
S1 detector at every tolerance. Matched-peak timing errors are accumulated for
the histogram, and the best example is remembered for the overlay panel.

In [ ]:
recon_rows = {t: [] for t in TOLS_MS}   # per-segment reconstruction metrics
pcg_rows   = {t: [] for t in TOLS_MS}   # per-segment PCG-only-baseline metrics
seg_meta   = []                          # subject / scenario / stem per segment
all_timing_err_ms = []                   # signed matched-peak errors (recon, 150 ms)
recon_index = {}                         # stem -> {subject, scenario, path}

for path in tqdm(test_chunks, desc="segments"):
    stem = os.path.splitext(os.path.basename(path))[0]
    try:
        d = np.load(path, allow_pickle=True)
        ecg_raw = d['ecg'].astype(np.float32)
        pcg_raw = d['pcg'].astype(np.float32)
        subject = str(d['subject_id'])
        scen_idx = config_paths.scenario_to_index(d['scenario'])
    except Exception as e:
        print(f"[warn] skip {stem}: {e}")
        continue

    # Paper pipeline: polarity-fix the ECG target, z-score both model signals.
    ecg_z = metrics.normalize_zscore(metrics.ensure_positive_polarity(ecg_raw))
    pcg_z = metrics.normalize_zscore(pcg_raw)

    # Reconstruction: reuse cache if present, else run the generator.
    recon_path = os.path.join(RECON_DIR, f"{stem}.npy")
    if os.path.exists(recon_path):
        recon_z = np.load(recon_path).astype(np.float32)
    elif model is not None:
        pcg_t = torch.from_numpy(pcg_z).unsqueeze(0).unsqueeze(0).to(DEVICE)
        onehot = np.zeros(config_paths.NUM_SCENARIOS, dtype=np.float32)
        onehot[scen_idx] = 1.0
        scen_t = torch.from_numpy(onehot).unsqueeze(0).to(DEVICE)
        recon_raw = models.run_generator(model, pcg_t, scen_t, is_proposed=True)
        recon_z = metrics.normalize_zscore(recon_raw)   # z-score for scale-consistent detection
        np.save(recon_path, recon_z.astype(np.float32))
    else:
        print(f"[warn] no cache and no model for {stem}; skipping.")
        continue

    recon_index[stem] = {"subject": subject, "scenario": int(scen_idx),
                         "path": recon_path}
    seg_meta.append({"stem": stem, "subject": subject, "scenario": int(scen_idx)})

    # Ground-truth and PCG-only S1 peaks (shared once per segment).
    true_pk = metrics.detect_r_peaks(ecg_z, fs=FS, distance=DIST, prominence=PROM)
    s1_pk, _ = detect_s1_peaks(pcg_raw, FS)

    for tol in TOLS_MS:
        recon_rows[tol].append(
            metrics.temporal_rpeak_metrics(ecg_z, recon_z, fs=FS, tol_ms=tol,
                                           distance=DIST, prominence=PROM))
        pcg_rows[tol].append(
            temporal_metrics_from_peaks(true_pk, s1_pk, FS, tol))

    # Accumulate matched-peak timing errors for the histogram (recon, +/-150 ms).
    recon_pk = metrics.detect_r_peaks(recon_z, fs=FS, distance=DIST, prominence=PROM)
    _, _, _, _, terr = metrics.match_peaks(
        true_pk, recon_pk, int(round(150 / 1000 * FS)))
    all_timing_err_ms.extend((terr / FS * 1000.0).tolist())

with open(os.path.join(RECON_DIR, "_index.json"), "w") as f:
    json.dump(_sanitize(recon_index), f, indent=2)
print(f"[done] scored {len(seg_meta)} segments; "
      f"cached reconstructions in {RECON_DIR}")

## Cell 5 -- aggregate overall, per-subject, per-scenario

Detection totals pool TP/FP/FN before Se/PPV/F1 are recomputed (the correct way
to aggregate detection over segments). Timing, RR, and HR errors average over
the segments where they are defined, with the valid-N reported.

In [ ]:
SCEN_NAME = {0: "rest", 1: "exercise", 2: "recovery"}

def _subset(rows, keep):
    return [r for r, m in zip(rows, seg_meta) if keep(m)]

results = {"n_segments": len(seg_meta), "tolerances_ms": list(TOLS_MS),
           "overall": {}, "per_subject": {}, "per_scenario": {}}

for tol in TOLS_MS:
    results["overall"][f"recon_{tol}ms"] = metrics.aggregate_temporal(recon_rows[tol])
    results["overall"][f"pcg_{tol}ms"]   = metrics.aggregate_temporal(pcg_rows[tol])

subjects  = sorted({m["subject"] for m in seg_meta})
scenarios = sorted({m["scenario"] for m in seg_meta})

for subj in subjects:
    results["per_subject"][subj] = {}
    for tol in TOLS_MS:
        results["per_subject"][subj][f"recon_{tol}ms"] = metrics.aggregate_temporal(
            _subset(recon_rows[tol], lambda m, s=subj: m["subject"] == s))
        results["per_subject"][subj][f"pcg_{tol}ms"] = metrics.aggregate_temporal(
            _subset(pcg_rows[tol], lambda m, s=subj: m["subject"] == s))

for sc in scenarios:
    name = SCEN_NAME.get(sc, str(sc))
    results["per_scenario"][name] = {}
    for tol in TOLS_MS:
        results["per_scenario"][name][f"recon_{tol}ms"] = metrics.aggregate_temporal(
            _subset(recon_rows[tol], lambda m, c=sc: m["scenario"] == c))
        results["per_scenario"][name][f"pcg_{tol}ms"] = metrics.aggregate_temporal(
            _subset(pcg_rows[tol], lambda m, c=sc: m["scenario"] == c))

results["n_matched_peaks_hist"] = len(all_timing_err_ms)

with open(os.path.join(OUT_DIR, "results.json"), "w") as f:
    json.dump(_sanitize(results), f, indent=2)
print(f"[done] wrote {os.path.join(OUT_DIR, 'results.json')}")

# Console summary
print("\n===== OVERALL (Se / PPV / F1) =====")
for tol in TOLS_MS:
    r = results["overall"][f"recon_{tol}ms"]
    p = results["overall"][f"pcg_{tol}ms"]
    print(f"  +/-{tol:>3} ms  recon: Se {r['sensitivity']:.3f}  PPV {r['ppv']:.3f}  "
          f"F1 {r['f1']:.3f}   |   PCG-only: Se {p['sensitivity']:.3f}  "
          f"PPV {p['ppv']:.3f}  F1 {p['f1']:.3f}")

## Cell 6 -- LaTeX tables

Main table: reconstruction vs PCG-only S1, at both tolerances, with the full
metric set. Per-subject / per-scenario table: F1 and median timing broken down.

In [ ]:
def _fmt(v, nd=3):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return "--"
    return f"{v:.{nd}f}"


def _row(label, agg):
    """Build one LaTeX row from an aggregate_temporal dict."""
    return (f"{label} & {_fmt(agg['sensitivity'])} & {_fmt(agg['ppv'])} & "
            f"{_fmt(agg['f1'])} & {agg['FP']} & {agg['FN']} & "
            f"{_fmt(agg['mae_timing_ms_mean'],1)} & "
            f"{_fmt(agg['median_timing_ms_mean'],1)} & "
            f"{_fmt(agg['iqr_timing_ms_mean'],1)} & "
            f"{_fmt(agg['rr_error_ms_mean'],1)} & "
            f"{_fmt(agg['hr_error_bpm_mean'],1)} \\\\")


tbl = []
tbl.append("% Auto-generated by exp1_R1_1_rpeak_timing.py -- do not edit by hand.")
tbl.append("% Paste into the manuscript; answers Reviewer 1, comment 1 (R1.1).")
tbl.append(r"\begin{table*}[t]")
tbl.append(r"\centering")
tbl.append(r"\caption{Temporal R-peak detection on the EPHNOGRAM test set. "
           r"Peaks are detected independently in each signal and matched to the "
           r"ground-truth R-peaks. The reconstructed ECG is compared against a "
           r"PCG-only S1 envelope detector at two tolerances. Timing statistics "
           r"average the per-segment mean, median, and IQR of matched-peak error.}")
tbl.append(r"\label{tab:rpeak_timing}")
tbl.append(r"\small")
tbl.append(r"\begin{tabular}{l l c c c c c c c c c c}")
tbl.append(r"\toprule")
tbl.append(r"Method & Tol. & Se & PPV & F1 & FP & FN & "
           r"Mean & Median & IQR & RR err. & HR err. \\")
tbl.append(r" &  &  &  &  &  &  & (ms) & (ms) & (ms) & (ms) & (bpm) \\")
tbl.append(r"\midrule")
for tol in TOLS_MS:
    tbl.append(_row(rf"Reconstructed ECG (Proposed) & $\pm${tol}\,ms",
                    results["overall"][f"recon_{tol}ms"]))
for tol in TOLS_MS:
    tbl.append(_row(rf"PCG-only S1 envelope & $\pm${tol}\,ms",
                    results["overall"][f"pcg_{tol}ms"]))
tbl.append(r"\bottomrule")
tbl.append(r"\end{tabular}")
tbl.append(r"\end{table*}")

with open(os.path.join(OUT_DIR, "exp1_temporal_table.tex"), "w") as f:
    f.write("\n".join(tbl) + "\n")
print(f"[done] wrote {os.path.join(OUT_DIR, 'exp1_temporal_table.tex')}")


# Per-subject / per-scenario table (reconstruction).
def _brk_row(label, meta_agg50, agg50, agg150):
    return (f"{label} & {meta_agg50['n_segments']} & {agg50['TP'] + agg50['FN']} & "
            f"{_fmt(agg50['sensitivity'])} & {_fmt(agg50['ppv'])} & "
            f"{_fmt(agg50['f1'])} & {_fmt(agg150['f1'])} & "
            f"{_fmt(agg50['median_timing_ms_mean'],1)} \\\\")


ptbl = []
ptbl.append("% Auto-generated by exp1_R1_1_rpeak_timing.py -- do not edit by hand.")
ptbl.append(r"\begin{table}[t]")
ptbl.append(r"\centering")
ptbl.append(r"\caption{Per-subject and per-scenario temporal R-peak detection for "
            r"the reconstructed ECG. F1 is reported at $\pm$50\,ms and $\pm$150\,ms; "
            r"the median column is the mean per-segment median timing error.}")
ptbl.append(r"\label{tab:rpeak_timing_breakdown}")
ptbl.append(r"\small")
ptbl.append(r"\begin{tabular}{l c c c c c c c}")
ptbl.append(r"\toprule")
ptbl.append(r"Group & Seg. & R-peaks & Se & PPV & F1$_{50}$ & F1$_{150}$ & Med. (ms) \\")
ptbl.append(r"\midrule")
ptbl.append(r"\multicolumn{8}{l}{\textit{Per subject}} \\")
for subj in subjects:
    a50  = results["per_subject"][subj]["recon_50ms"]
    a150 = results["per_subject"][subj]["recon_150ms"]
    ptbl.append(_brk_row(subj, a50, a50, a150))
ptbl.append(r"\midrule")
ptbl.append(r"\multicolumn{8}{l}{\textit{Per scenario}} \\")
for sc in scenarios:
    name = SCEN_NAME.get(sc, str(sc))
    a50  = results["per_scenario"][name]["recon_50ms"]
    a150 = results["per_scenario"][name]["recon_150ms"]
    ptbl.append(_brk_row(name.capitalize(), a50, a50, a150))
ptbl.append(r"\midrule")
o50, o150 = results["overall"]["recon_50ms"], results["overall"]["recon_150ms"]
ptbl.append(_brk_row(r"\textbf{Overall}", o50, o50, o150))
ptbl.append(r"\bottomrule")
ptbl.append(r"\end{tabular}")
ptbl.append(r"\end{table}")

with open(os.path.join(OUT_DIR, "exp1_persubject_table.tex"), "w") as f:
    f.write("\n".join(ptbl) + "\n")
print(f"[done] wrote {os.path.join(OUT_DIR, 'exp1_persubject_table.tex')}")

## Cell 7 -- double-column figure (overlay + timing-error histogram)

Panel (a): a representative segment with ground-truth vs reconstructed ECG, the
detected GT peaks (circles), the reconstructed peaks (crosses), and the
+/-50 ms tolerance band around each GT peak. Panel (b): the pooled histogram of
matched-peak timing error.

In [ ]:
# Pick the clearest example: best F1 at +/-50 ms among segments with >= 5 GT peaks.
best_i, best_key = -1, (-1.0, -1)
for i, r in enumerate(recon_rows[50]):
    if r["n_true"] >= 5 and r["f1"] is not None and not np.isnan(r["f1"]):
        key = (r["f1"], r["n_true"])
        if key > best_key:
            best_key, best_i = key, i
if best_i < 0:  # fall back to any segment with the most GT peaks
    best_i = int(np.argmax([r["n_true"] for r in recon_rows[50]])) if recon_rows[50] else -1

fig, axes = ieee_style.new_fig("double", height=2.7, nrows=1, ncols=2)
ax_a, ax_b = axes[0], axes[1]

if best_i >= 0:
    meta = seg_meta[best_i]
    stem = meta["stem"]
    ecg_z = metrics.normalize_zscore(
        metrics.ensure_positive_polarity(
            np.load(os.path.join(chunk_dir, f"{stem}.npz"),
                    allow_pickle=True)['ecg'].astype(np.float32)))
    recon_z = np.load(os.path.join(RECON_DIR, f"{stem}.npy")).astype(np.float32)
    t_pk = metrics.detect_r_peaks(ecg_z, fs=FS, distance=DIST, prominence=PROM)
    r_pk = metrics.detect_r_peaks(recon_z, fs=FS, distance=DIST, prominence=PROM)

    # 4 s window starting a little before the first GT peak, for legibility.
    win = int(4.0 * FS)
    start = max(0, int(t_pk[0]) - int(0.3 * FS)) if len(t_pk) else 0
    end = min(len(ecg_z), start + win)
    t = np.arange(start, end) / FS
    tol_s = 50 / 1000.0

    ax_a.plot(t, ecg_z[start:end], color=ieee_style.COLORS["truth"],
              lw=1.0, label="Ground truth")
    ax_a.plot(t, recon_z[start:end], color=ieee_style.COLORS["proposed"],
              lw=1.0, alpha=0.9, label="Reconstructed")
    band_lbl = r"$\pm$50 ms"
    for pk in t_pk:
        if start <= pk < end:
            ax_a.axvspan(pk / FS - tol_s, pk / FS + tol_s, color="0.6",
                         alpha=0.15, lw=0, label=band_lbl)
            band_lbl = None  # label only once
    gt_lbl, rc_lbl = "GT R-peak", "Recon R-peak"
    for pk in t_pk:
        if start <= pk < end:
            ax_a.plot(pk / FS, ecg_z[pk], 'o', mfc='none',
                      mec=ieee_style.COLORS["truth"], mew=1.2, ms=6, label=gt_lbl)
            gt_lbl = None
    for pk in r_pk:
        if start <= pk < end:
            ax_a.plot(pk / FS, recon_z[pk], 'x',
                      color=ieee_style.COLORS["proposed"], mew=1.2, ms=6, label=rc_lbl)
            rc_lbl = None
    ieee_style.finalize_axes(
        ax_a, xlabel="Time (s)", ylabel="Amplitude (z-score, a.u.)",
        title=f"(a) {meta['subject']}, {SCEN_NAME.get(meta['scenario'], '')}",
        legend=True, legend_loc="upper right", legend_ncol=2)
else:
    ax_a.text(0.5, 0.5, "no example segment", ha="center", va="center")

if all_timing_err_ms:
    errs = np.asarray(all_timing_err_ms)
    ax_b.hist(errs, bins=np.arange(-150, 151, 10),
              color=ieee_style.COLORS["proposed"], edgecolor="white", linewidth=0.4)
    ax_b.axvline(0.0, color="0.3", lw=0.9, ls="--")
    ieee_style.finalize_axes(
        ax_b, xlabel="Matched-peak timing error (ms)", ylabel="Count",
        title=f"(b) matched peaks (n = {len(errs)})")
else:
    ax_b.text(0.5, 0.5, "no matched peaks", ha="center", va="center")

ieee_style.save_fig(fig, OUT_DIR, "fig_exp1_overlay_hist")

## Cell 8 -- single-column per-subject F1 bar chart

Reconstruction vs PCG-only S1, F1 at +/-50 ms, grouped by subject with an
overall pair.

In [ ]:
labels = subjects + ["Overall"]
recon_f1 = [results["per_subject"][s]["recon_50ms"]["f1"] for s in subjects]
recon_f1.append(results["overall"]["recon_50ms"]["f1"])
pcg_f1 = [results["per_subject"][s]["pcg_50ms"]["f1"] for s in subjects]
pcg_f1.append(results["overall"]["pcg_50ms"]["f1"])

recon_f1 = [0.0 if (v is None or np.isnan(v)) else v for v in recon_f1]
pcg_f1   = [0.0 if (v is None or np.isnan(v)) else v for v in pcg_f1]

x = np.arange(len(labels))
w = 0.38
fig2, ax = ieee_style.new_fig("single", height=2.4)
ax.bar(x - w / 2, recon_f1, w, color=ieee_style.COLORS["proposed"],
       label="Reconstructed ECG")
ax.bar(x + w / 2, pcg_f1, w, color=ieee_style.COLORS["pcg"],
       label="PCG-only S1")
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=0)
ax.set_ylim(0, 1.0)
ieee_style.finalize_axes(ax, xlabel="Subject", ylabel="R-peak F1 ($\\pm$50 ms)",
                         legend=True, legend_loc="upper right")
ieee_style.save_fig(fig2, OUT_DIR, "fig_exp1_persubject_f1")

print("\n[all done] exp1 temporal R-peak metrics complete.")
print(f"  results : {os.path.join(OUT_DIR, 'results.json')}")
print(f"  tables  : exp1_temporal_table.tex, exp1_persubject_table.tex")
print(f"  figures : fig_exp1_overlay_hist.(pdf|png), fig_exp1_persubject_f1.(pdf|png)")
print(f"  recons  : {RECON_DIR}  (reused by exp3)")